# Sistema Multiagente RAG — AEM3 / PIM3 (i2T)

Este notebook ejecuta el flujo completo del sistema:

1. Setup e imports.
2. Construcción/carga de los índices RAG por dominio (LangChain + FAISS).
3. Definición del grafo de routing (LangGraph).
4. Ejecución de consultas de ejemplo, con tracing en Langfuse.
5. Validación contra `test_queries.json`.
6. (Bonus) Evaluación automática de calidad con la Score API de Langfuse.


In [ ]:
import sys
sys.path.insert(0, "..")

from src.config import DOMAIN_PATHS, LANGFUSE_ENABLED
from src.rag import RAGRegistry
from src.graph import compiled_graph
from src.langfuse_setup import get_langfuse_handler
from src.evaluator import evaluate_result

print("Dominios configurados:", list(DOMAIN_PATHS.keys()))
print("Langfuse habilitado:", LANGFUSE_ENABLED)


## 1. Construcción de los índices RAG

La primera ejecución calcula los embeddings y construye el índice FAISS de
cada dominio; las siguientes ejecuciones reutilizan el índice persistido en
`.vectorstores/` si los documentos fuente no cambiaron.

In [ ]:
registry = RAGRegistry()
registry.build_all()
print("Índices listos para:", list(registry._vectorstores.keys()))


## 2. Ejecutar una consulta a través del grafo completo

`compiled_graph.invoke` corre: orquestador -> routing condicional -> agente
de dominio (o `unknown`), devolviendo el estado final con la respuesta, las
fuentes usadas y la intención detectada.

In [ ]:
handler = get_langfuse_handler()
config = {"callbacks": [handler]} if handler else {}

query = "¿Cuántos días de vacaciones me corresponden con 6 años de antigüedad?"
result = compiled_graph.invoke({"query": query}, config=config)

print("Intención detectada:", result["intent"], "-", result["reason"])
print("Fuentes:", result["sources"])
print()
print("Respuesta:")
print(result["answer"])


## 3. Probar los tres dominios y el caso `unknown`

In [ ]:
ejemplos = [
    "No puedo conectarme a la VPN desde mi notebook",
    "¿Cuándo se paga el reembolso de una factura de viáticos ya aprobada?",
    "¿Cuál es la capital de Francia?",
]

for q in ejemplos:
    r = compiled_graph.invoke({"query": q}, config=config)
    print(f"[{r['intent']}] {q}")
    print(" ->", r["answer"][:200], "...")
    print()


## 4. Validar el routing contra `test_queries.json`

In [ ]:
import json
from src.config import TEST_QUERIES_PATH

test_cases = json.loads(TEST_QUERIES_PATH.read_text(encoding="utf-8"))
correct = 0

for case in test_cases:
    r = compiled_graph.invoke({"query": case["query"]}, config=config)
    ok = r["intent"] == case["expected_intent"]
    correct += int(ok)
    print(f"[{'OK  ' if ok else 'FAIL'}] esperado={case['expected_intent']:<8} obtenido={r['intent']:<8} | {case['query']}")

print()
print(f"Precisión de routing: {correct}/{len(test_cases)} ({correct/len(test_cases):.0%})")


## 5. (Bonus) Evaluación automática de calidad

Se usa un LLM como juez para puntuar `groundedness` y `relevance` de una
respuesta, y se envían los puntajes a Langfuse mediante la Score API
(requiere tener LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY configurados en
`.env` para que el envío del score sea efectivo).

In [ ]:
query = "¿Cómo cambio la contraseña de mi correo corporativo?"
result = compiled_graph.invoke({"query": query}, config=config)

evaluacion = evaluate_result(result)  # trace_id opcional si se quiere asociar a Langfuse
print(json.dumps(evaluacion, indent=2, ensure_ascii=False))
